# M1: Ingesta TCGA-BRCA - GEN-MAMA-MX

Este notebook automatiza la descarga y procesamiento de datos de mutaciones somáticas del proyecto **TCGA-BRCA** (Breast Invasive Carcinoma) utilizando la API del **GDC (Genomic Data Commons)**.

**Objetivos:**
1. Descargar datos de mutaciones somáticas filtrados para variantes missense.
2. Identificar los genes más frecuentemente mutados.
3. Priorizar las variantes para el análisis estructural (Módulo M3).

In [ ]:
import pandas as pd
import requests
import json
import os
from tqdm import tqdm

print("Librerías cargadas correctamente.")

## 1. Configuración de la API del GDC

Definimos los parámetros para consultar el portal GDC y obtener las mutaciones somáticas (SSM) del proyecto BRCA.

In [ ]:
def fetch_ssms(project_id="TCGA-BRCA", size=1000):
    """Consulta la API de GDC para obtener Simple Somatic Mutations."""
    ssm_url = "https://api.gdc.cancer.gov/ssms"
    
    # Filtros: Proyecto TCGA-BRCA y Variantes Missense
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {
                    "field": "occurrence.case.project.project_id",
                    "value": [project_id]
                }
            },
            {
                "op": "in",
                "content": {
                    "field": "consequence.transcript.consequence_type",
                    "value": ["missense_variant"]
                }
            }
        ]
    }
    
    params = {
        "filters": json.dumps(filters),
        "fields": "genomic_dna_change,consequence.transcript.gene.symbol,consequence.transcript.aa_change,occurrence.case.project.project_id",
        "format": "JSON",
        "size": size
    }
    
    response = requests.get(ssm_url, params=params)
    return response.json()

print("Iniciando descarga de datos de GDC...")
data = fetch_ssms(size=2000)
print(f"Descarga completada. Registros obtenidos: {len(data['data']['hits'])}")

## 2. Procesamiento y Limpieza de Datos

Transformamos el JSON en un DataFrame de Pandas para su manipulación.

In [ ]:
hits = data['data']['hits']
rows = []

for hit in hits:
    gene_symbol = hit['consequence'][0]['transcript']['gene']['symbol']
    aa_change = hit['consequence'][0]['transcript'].get('aa_change', 'N/A')
    dna_change = hit['genomic_dna_change']
    
    rows.append({
        "Gene": gene_symbol,
        "AA_Change": aa_change,
        "DNA_Change": dna_change
    })

df = pd.DataFrame(rows)
df.dropna(inplace=True)

# Guardar dataset completo
df.to_csv("mutations_brca.csv", index=False)
df.head(10)

## 3. Priorización de Variantes (M2)

Calculamos la frecuencia de las mutaciones para identificar los hotspots más relevantes del proyecto.

In [ ]:
top_variants = df.groupby(['Gene', 'AA_Change']).size().reset_index(name='Frequency')
top_variants = top_variants.sort_values(by='Frequency', ascending=False).head(10)

print("Top 10 Variantes Missense en TCGA-BRCA:")
print(top_variants)

# Guardar resultados para el siguiente módulo
top_variants_dict = top_variants.to_dict(orient='records')
with open("top_variants.json", "w") as f:
    json.dump(top_variants_dict, f, indent=4)

print("\nArchivos 'mutations_brca.csv' y 'top_variants.json' generados.")